In [6]:
# Task 10: Performance Optimization (Mandatory)
# Partition Strategy
# Repartition data by:
# 1. Date
# 2. Country/Region
# Explain difference between:
# * repartition()
# * coalesce()
# Apply partitioning before writing Parquet.

# Data Skew Handling
# 1. Identify skewed countries (e.g., USA).
# 2. Implement salting technique OR skew join optimization.
# 3. Explain impact of skew on shuffle.

# Broadcast Join
# 1. Use broadcast join when joining worldometer_data.
# 2. Verify using: df.explain("formatted")
# 3. Confirm BroadcastHashJoin is used.

# Shuffle Optimization
# 1. Tune: spark.sql.shuffle.partitions
# 2. Avoid unnecessary wide transformations.
# 3. Combine filters before joins.
# Explain why shuffle is expensive.

# Caching Strategy
# 1. Cache frequently reused DataFrames.
# 2. Use persist(StorageLevel.MEMORY_AND_DISK).
# 3. Explain when caching degrades performance.

In [24]:
#---------------------------------------------------------------------------
# Import Statements
#---------------------------------------------------------------------------
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

In [8]:
#---------------------------------------------------------------------------
# Starting Spark Session
#---------------------------------------------------------------------------
spark = SparkSession.builder \
    .appName("Performance Optimization") \
    .master("yarn") \
    .getOrCreate()

In [14]:

#---------------------------------------------------------------------------
# Define Paths
#---------------------------------------------------------------------------
STAGING_PATH = "hdfs:///data/covid/staging/"
ANALYTICS_PATH = "hdfs:///data/covid/analytics/"

#---------------------------------------------------------------------------
# Read Data
#---------------------------------------------------------------------------
worldometer_data = spark.read.parquet(STAGING_PATH + "worldometer_data")
full_grouped = spark.read.parquet(STAGING_PATH + "full_grouped")

In [20]:
#===========================================================================
# 1. Partition Strategy
#===========================================================================

# Repartition by Date
full_grouped_by_date = full_grouped.repartition("Date")

# Repartition by Country/Region
worldometer_by_country = worldometer_data.repartition("Country/Region")

# Write partitioned parquet
worldometer_by_country.write \
    .mode("overwrite") \
    .partitionBy("Country/Region") \
    .parquet(ANALYTICS_PATH + "partitioned_by_country")

full_grouped_by_date.write \
    .mode("overwrite") \
    .partitionBy("Date") \
    .parquet(ANALYTICS_PATH + "full_grouped_by_date")
# Explanation:
# repartition() → Full shuffle, increases/decreases partitions
# coalesce() → Reduces partitions without full shuffle (narrow transformation)


In [21]:
#===========================================================================
# 2. Data Skew Handling
#===========================================================================
# Identify skewed countries
country_distribution = worldometer_data.groupBy("Country/Region") \
    .count() \
    .orderBy(col("count").desc())

country_distribution.show()

# Salting Technique
salted_df = worldometer_data.withColumn(
    "salt",
    floor(rand() * 5)
)

# Example salted aggregation
salted_agg = salted_df.groupBy("Country/Region", "salt") \
    .agg(sum("TotalCases").alias("salted_sum")) \
    .groupBy("Country/Region") \
    .agg(sum("salted_sum").alias("final_sum"))

salted_agg.show()

# Impact of Skew:
# - Uneven partition sizes
# - Long shuffle stages
# - Straggler tasks
# - Memory spill
# - Increased job completion time


+--------------------+-----+
|      Country/Region|count|
+--------------------+-----+
|              Panama|    1|
|              Canada|    1|
|           Hong Kong|    1|
|                Chad|    1|
|             Brunei |    1|
|              Russia|    1|
|          Madagascar|    1|
|            Paraguay|    1|
|             Albania|    1|
|             Ukraine|    1|
|           Venezuela|    1|
|             Finland|    1|
|          Seychelles|    1|
|Caribbean Netherl...|    1|
|              Israel|    1|
|          Kyrgyzstan|    1|
|          Uzbekistan|    1|
|     North Macedonia|    1|
|             Iceland|    1|
|               Macao|    1|
+--------------------+-----+
only showing top 20 rows
+--------------------+---------+
|      Country/Region|final_sum|
+--------------------+---------+
|              Panama|    71418|
|           Hong Kong|     3850|
|              Canada|   118561|
|                Chad|      942|
|             Brunei |      141|
|              

In [26]:
#===========================================================================
# 3. Broadcast Join Optimization
#===========================================================================
small_df = worldometer_data.select("Country/Region", "Population")

joined_df = full_grouped.join(
    broadcast(small_df),
    on="Country/Region",
    how="inner"
)

joined_df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (11)
+- Project (10)
   +- BroadcastHashJoin Inner BuildRight (9)
      :- Filter (2)
      :  +- Scan parquet  (1)
      +- BroadcastExchange (8)
         +- Filter (7)
            +- InMemoryTableScan (3)
                  +- InMemoryRelation (4)
                        +- * ColumnarToRow (6)
                           +- Scan parquet  (5)


(1) Scan parquet 
Output [10]: [Date#182, Country/Region#183, Confirmed#184L, Deaths#185L, Recovered#186L, Active#187L, New cases#188L, New deaths#189L, New recovered#190L, WHO Region#191]
Batched: true
Location: InMemoryFileIndex [hdfs://localhost:9000/data/covid/staging/full_grouped]
PushedFilters: [IsNotNull(`Country/Region`)]
ReadSchema: struct<Date:date,Country/Region:string,Confirmed:bigint,Deaths:bigint,Recovered:bigint,Active:bigint,New cases:bigint,New deaths:bigint,New recovered:bigint,WHO Region:string>

(2) Filter
Input [10]: [Date#182, Country/Region#183, Confirmed#184L, Deaths#185L, Recovered#18

In [27]:
#===========================================================================
# 4. Shuffle Optimization
#===========================================================================
# Tune shuffle partitions
spark.conf.set("spark.sql.shuffle.partitions", 50)

# Combine filters before join
filtered_full_grouped = full_grouped.filter(col("Confirmed") > 1000)

optimized_join = filtered_full_grouped.join(
    worldometer_data,
    on="Country/Region",
    how="inner"
)

optimized_join.explain()

# Why shuffle is expensive:
# - Disk I/O
# - Network transfer
# - Serialization/deserialization
# - Spill to disk
# - Stage barriers


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [Country/Region#183, Date#182, Confirmed#184L, Deaths#185L, Recovered#186L, Active#187L, New cases#188L, New deaths#189L, New recovered#190L, WHO Region#191, Continent#163, Population#164L, TotalCases#165L, NewCases#166L, TotalDeaths#167L, NewDeaths#168L, TotalRecovered#169L, NewRecovered#170L, ActiveCases#171L]
   +- BroadcastHashJoin [Country/Region#183], [Country/Region#162], Inner, BuildRight, false
      :- Filter ((isnotnull(Confirmed#184L) AND (Confirmed#184L > 1000)) AND isnotnull(Country/Region#183))
      :  +- FileScan parquet [Date#182,Country/Region#183,Confirmed#184L,Deaths#185L,Recovered#186L,Active#187L,New cases#188L,New deaths#189L,New recovered#190L,WHO Region#191] Batched: true, DataFilters: [isnotnull(Confirmed#184L), (Confirmed#184L > 1000), isnotnull(Country/Region#183)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/data/covid/staging/full_grouped], PartitionFilters:

In [28]:
#===========================================================================
# 5. Caching Strategy
#===========================================================================
# Cache frequently reused DataFrame
worldometer_data.persist(StorageLevel.MEMORY_AND_DISK)

# Trigger action
worldometer_data.count()

# When caching degrades performance:
# - Data too large for memory
# - Cached but rarely reused
# - Causes GC pressure
# - Evicts useful cached data


208

In [29]:

#---------------------------------------------------------------------------
# Stop Spark
#---------------------------------------------------------------------------
spark.stop()